In [0]:
# %pip install azure-eventhub

# # Restart Python runtime after install
# dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone
import json
import requests
import time
from azure.eventhub import EventData, EventHubProducerClient, TransportType

# Parameters
secret_scope = "valerii-matviiv-scope"
secret_key_finnhub = "finnhub-api-key"
secret_key_eventhub = "valerii-eventhub-cs"
eventhub_name = "valeriimatviiv_evh"  # Your new Event Hub instance
tickers = ["AAPL", "NVDA", "MSFT", "AMZN", "TSLA", "QQQ"]
event_count = 20

# Secrets
finnhub_key = dbutils.secrets.get(scope=secret_scope, key=secret_key_finnhub)
eh_conn_str = dbutils.secrets.get(
    scope=secret_scope, key=secret_key_eventhub
).strip()

In [0]:
def fetch_stock_quote(symbol: str) -> dict:
    url = f"https://finnhub.io/api/v1/quote?symbol={symbol}&token={finnhub_key}"
    res = requests.get(url)
    if res.status_code == 200:
        d = res.json()
        return {
            "symbol": symbol,
            "current_price": float(d.get("c", 0.0)),
            "high_price": float(d.get("h", 0.0)),
            "low_price": float(d.get("l", 0.0)),
            "open_price": float(d.get("o", 0.0)),
            "previous_close": float(d.get("pc", 0.0)),
            "event_timestamp": datetime.now(timezone.utc).isoformat(),
        }
    return None


producer = EventHubProducerClient.from_connection_string(
    conn_str=eh_conn_str,
    eventhub_name=eventhub_name,
    transport_type=TransportType.AmqpOverWebsocket,
)

print(f"Publishing live price ticks to '{eventhub_name}'...")

with producer:
    for i in range(event_count):
        symbol = tickers[i % len(tickers)]
        quote = fetch_stock_quote(symbol)
        if quote:
            batch = producer.create_batch()
            batch.add(EventData(json.dumps(quote)))
            producer.send_batch(batch)
            print(
                f"[{i + 1}/{event_count}] Sent tick for {symbol} | Price: ${quote['current_price']}"
            )
        time.sleep(1)

print("Finished sending events.")